In [1]:
from modules.dataset.make_dataset import make_dataset, dataset_split_by_recording, stratified_recording_folds, NpzDataPipeline, AugmentedSubset, dataset_split
from modules.nn import DefaultCNN, BranchedCoughCNN
import polars as pl
import numpy as np
import torch
from torchinfo import summary
from torch.utils.data import Subset
import torch.nn as nn
import torch.optim as optim
from typing import Sequence
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [2]:
dataset: NpzDataPipeline = NpzDataPipeline("data\\final_processed_dataset.npz")
device: torch.types.Device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
import os

# ---- Crash-proof run state: per-epoch autosave + resume (train_model) ----
RUN_STATE_DIR: str = "run_state_3class"
os.makedirs(RUN_STATE_DIR, exist_ok=True)

def _ckpt_path(name: str) -> str:
    return os.path.join(RUN_STATE_DIR, f"{name}_latest.pth.tar")

def _atomic_torch_save(obj: dict, path: str) -> None:
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)  # crash mid-write never corrupts the good file

In [4]:
# ---- Experiment setup (shared) ----
from modules.dataset.make_dataset import AugmentedSubset, aug_probs_from_counts
from modules.preprocessing.audio_process import WaveformAugmentor, SpecAugmentations, MelSpectrogramPipeline, SpectMFCCPipeline, trim_tile
from tqdm.auto import tqdm

num_epochs: int = 60  # epoch cap for ALL runs; early stop (15/3) lives in train_model
N_FOLDS: int = 5
SEED: int = 42  # identical split across all experiments
BATCH_SIZE: int = 128

# n_mel=64 per Chang et al. (research_results.md fix #6): 64 mel bins @ 16 kHz
mel_train_plain: SpectMFCCPipeline = SpectMFCCPipeline(combind=True).to(device)  # no spec aug
mel_eval: SpectMFCCPipeline = SpectMFCCPipeline(combind=True).to(device)         # plain for validation

# speed: compile only the eval mel path; train paths stay eager (spec-aug masks/augmentor randomize per step)
mel_eval = torch.compile(mel_eval) if device.type == "cuda" else mel_eval  # CPU inductor needs MSVC cl.exe; skip
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

In [5]:
# ---- Recording-wise 5-fold CV setup (replaces the single 80/10/10 split) ----
# Recording-grouped, class-stratified folds. The old test split survives only as
# the pool definition: train+val recordings form the CV pool, test stays held out.
# Every reported metric comes from pooled out-of-fold predictions, aggregated to
# the RECORDING level before anything is computed.

dataset.load_data()
dataset.load_paths()

train_subset, val_subset, test_subset = dataset_split(dataset, 0.8, 0.1, seed=SEED)

# npz label order (never assume): 0=healthy, 1=COVID-19, 2=symptomatic
LABELS_INV = {0: "healthy", 1: "COVID-19", 2: "symptomatic"}
CLASS_NAMES: list[str] = [LABELS_INV[i] for i in range(3)]
print("label map:", LABELS_INV)

cv_pool: list[int] = [i for s in (train_subset, val_subset) for i in s.indices]

# recording index: path -> pool positions; recordings are single-label (npz blocks are per class)
rec_idx: dict[str, list[int]] = {}
for p in cv_pool:
    rec_idx.setdefault(dataset.paths[p], []).append(p)
rec_label_of: dict[str, int] = {}
for path, idxs in rec_idx.items():
    labs = {int(dataset.labels[i]) for i in idxs}
    assert len(labs) == 1, f"recording {path} has mixed labels {labs} — grouping assumption broken"
    rec_label_of[path] = next(iter(labs))

cv_labels: list[int] = [rec_label_of[dataset.paths[i]] for i in cv_pool]
FOLDS: list[tuple[list[int], list[int]]] = stratified_recording_folds(
    [dataset.paths[i] for i in cv_pool], cv_labels, n_splits=N_FOLDS, seed=SEED
)

# ---- Oversampling: repeat minority-class train indices until every class matches ----
# the largest class's count (full balance; ~15x COVID / ~3.7x symptomatic naturally).
# TRAIN-side only — val keeps the natural distribution (duplicates there would
# inflate metrics). Repetition (not loss weighting) composes with augmentation:
# every repeated draw gets a fresh random augment pass.
def oversample_balance(positions: Sequence[int]) -> list[int]:
    by_cls: dict[int, list[int]] = {0: [], 1: [], 2: []}
    for p in positions:
        by_cls[int(dataset.labels[p])].append(p)
    natural = {k: len(v) for k, v in by_cls.items()}
    target = max(natural.values())
    out: list[int] = []
    for idxs in by_cls.values():
        if not idxs:
            continue
        reps, rem = divmod(target, len(idxs))
        out.extend(idxs * reps + idxs[:rem])
    np.random.default_rng(SEED).shuffle(out)
    print(f"oversample: {len(positions)} -> {len(out)} train positions "
          f"(natural {natural} -> all {target})")
    return out

# runnable check: leakage + stratification
_va_recs = [{dataset.paths[cv_pool[p]] for p in va_pos} for _, va_pos in FOLDS]
assert sorted(p for _, va in FOLDS for p in va) == list(range(len(cv_pool))), "folds don't partition the pool"
assert sum(len(rs) for rs in _va_recs) == len(rec_idx), "a recording landed in two val folds"
for fi, (tr_pos, va_pos) in enumerate(FOLDS):
    tr_recs = {dataset.paths[cv_pool[p]] for p in tr_pos}
    assert not (tr_recs & _va_recs[fi]), f"fold {fi}: a recording is in both train and val"
    assert {cv_labels[p] for p in va_pos} == {0, 1, 2}, f"fold {fi}: val fold lost a class"
    counts = np.bincount([cv_labels[p] for p in va_pos], minlength=3)
    print(f"fold {fi}: val class counts " + str({CLASS_NAMES[k]: int(v) for k, v in enumerate(counts)}))
print(f"pool: {len(cv_pool)} segments / {len(rec_idx)} recordings; test holdout: {len(test_subset.indices)} segments")

healthy: 20092 (73.8%)
COVID-19: 1267 (4.7%)
symptomatic: 5852 (21.5%)
total: 27211
label map: {0: 'healthy', 1: 'COVID-19', 2: 'symptomatic'}
fold 0: val class counts {'healthy': 3620, 'COVID-19': 227, 'symptomatic': 1051}
fold 1: val class counts {'healthy': 3620, 'COVID-19': 227, 'symptomatic': 1051}
fold 2: val class counts {'healthy': 3619, 'COVID-19': 227, 'symptomatic': 1052}
fold 3: val class counts {'healthy': 3620, 'COVID-19': 227, 'symptomatic': 1051}
fold 4: val class counts {'healthy': 3620, 'COVID-19': 226, 'symptomatic': 1051}
pool: 24489 segments / 11324 recordings; test holdout: 2722 segments


In [6]:
_model = BranchedCoughCNN(num_classes=3)
# combined pipeline: 64 mel + 40 MFCC = 104 freq bins, 3 channels (spec, mfcc, delta)
# 3-class head: backbone -> (B, 3) logits over (healthy, COVID-19, symptomatic)
summary(_model, depth=5, input_size=[(1, 3, 64, 32), (1, 3, 40, 32)])

Layer (type:depth-idx)                        Output Shape              Param #
BranchedCoughCNN                              [1, 3]                    --
├─ModuleList: 1-7                             --                        (recursive)
│    └─DepthwiseSeparableBlock: 2-1           [1, 32, 32, 16]           --
│    │    └─Sequential: 3-1                   [1, 32, 32, 16]           --
│    │    │    └─Conv2d: 4-1                  [1, 12, 64, 32]           36
│    │    │    └─BatchNorm2d: 4-2             [1, 12, 64, 32]           24
│    │    │    └─ReLU6: 4-3                   [1, 12, 64, 32]           --
│    │    │    └─Conv2d: 4-4                  [1, 12, 32, 16]           108
│    │    │    └─BatchNorm2d: 4-5             [1, 12, 32, 16]           24
│    │    │    └─ReLU6: 4-6                   [1, 12, 32, 16]           --
│    │    │    └─Conv2d: 4-7                  [1, 32, 32, 16]           384
│    │    │    └─BatchNorm2d: 4-8             [1, 32, 32, 16]           64
│    │   

In [7]:
_model = DefaultCNN(num_classes=3)
# combined pipeline: 64 mel + 40 MFCC = 104 freq bins, 3 channels (spec, mfcc, delta)
# 3-class head: backbone -> (B, 3) logits over (healthy, COVID-19, symptomatic)
summary(_model, input_size=(1, 3, 104, 32))

Layer (type:depth-idx)                        Output Shape              Param #
DefaultCNN                                    [1, 3]                    --
├─ModuleList: 1-10                            --                        (recursive)
│    └─DepthwiseSeparableBlock: 2-1           [1, 32, 52, 16]           --
│    │    └─Sequential: 3-1                   [1, 32, 52, 16]           738
├─ModuleList: 1-11                            --                        (recursive)
│    └─Sequential: 2-2                        [1, 32, 52, 16]           --
│    │    └─Conv2d: 3-2                       [1, 32, 52, 16]           96
│    │    └─BatchNorm2d: 3-3                  [1, 32, 52, 16]           64
├─Dropout2d: 1-3                              [1, 32, 52, 16]           --
├─ModuleList: 1-10                            --                        (recursive)
│    └─DepthwiseSeparableBlock: 2-3           [1, 64, 26, 8]            --
│    │    └─Sequential: 3-4                   [1, 64, 26, 8]       

In [31]:
# ---- 3-class softmax trainer: logits over (healthy, COVID-19, symptomatic) ----
def train_model(
    name: str,
    model_name: str = "default",
    num_epochs: int = num_epochs,
    patience: int = 5,
    min_epoch: int = 15,
    wave_aug: bool = True,
    spec_aug: bool = True,
) -> tuple[dict, dict]:
    """3-class softmax training on the CV cell's rebound train_subset/val_subset.
    Single aug config (both). Crash-proof: _latest autosaved every epoch, _best on
    val-loss improvement, early stop at `min_epoch` after `patience` flat epochs
    (epoch-15 guardrail). OOF predictions are ALWAYS collected from the best
    checkpoint, even on a fully-resumed run. Returns (history, oof) where
    oof = {path, true, probs} per val segment (softmax probs, npz label order)."""

    history: dict[str, list[float]] = {"train_loss": [], "train_acc": [], "train_F1": [], "train_auc": [],
                                       "val_loss": [], "val_acc": [], "val_F1": [], "val_auc": []}

    # unweighted CE: oversample_balance (below) already balances the train loader;
    # inverse-frequency weights would double-correct (COVID ~15x, squared).
    criterion: nn.Module = nn.CrossEntropyLoss()

    # oversample to full balance: repeat minority-class train indices until they
    # match healthy's count. Train-side only - val keeps the natural distribution.
    train_positions: list[int] = oversample_balance(train_subset.indices)

    use_sep: bool = (model_name == "branched")
    model: nn.Module = _make_model(model_name, num_classes=3)
    model = torch.compile(model).to(device) if device.type == "cuda" else model.to(device)

    optimizer: optim.AdamW = optim.AdamW(model.parameters(), lr=4.5e-4, weight_decay=1e-2)
    scaler: torch.amp.GradScaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))
    scheduler: optim.lr_scheduler.CosineAnnealingLR = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-6
    )

    epochs_done, history, best_val_loss, plateau = _resume_with_best(
        name, model, optimizer, scheduler, scaler, num_epochs)
    # fresh-template backfill guard: new-key history (old checkpoints predate these keys)
    for k in ("train_auc", "val_auc"):
        history.setdefault(k, []).extend([float("nan")] * (len(history["val_loss"]) - len(history.get(k, []))))

    # per-run aug pipelines: spec-aug only when spec_aug, waveform aug only when wave_aug
    mel_train = SpectMFCCPipeline(
        transforms=SpecAugmentations().to(device) if spec_aug else None,
        combind=not use_sep,
    ).to(device)
    print(f"[{name}] aug config: wave={'on' if wave_aug else 'OFF'} spec={'on' if spec_aug else 'OFF'}")

    train_loader: torch.utils.data.DataLoader = torch.utils.data.DataLoader(
        # wave_aug off = no augmentor at all (AugmentedSubset passes items through)
        AugmentedSubset(dataset, train_positions, WaveformAugmentor(16_000) if wave_aug else None),
        BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True,
    )
    val_loader: torch.utils.data.DataLoader = torch.utils.data.DataLoader(
        Subset(dataset, val_subset.indices), BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True,
    )

    def forward_features(waveforms: torch.Tensor, raw: torch.Tensor, mel):
        """Combined pipelines return one (B,3,F,T) image; separated return (spec, mfcc)."""
        waveforms = trim_tile(waveforms)  # stored segs are ~75% trailing zero-pad
        feats = mel(waveforms, raw)
        if use_sep:
            spec, mfcc = feats
            if spec.ndim == 3:
                spec, mfcc = spec.unsqueeze(0), mfcc.unsqueeze(0)
            return spec, mfcc
        if feats.ndim == 3:
            feats = feats.unsqueeze(0)
        return feats

    for epoch in range(epochs_done, num_epochs):
        model.train()
        running_loss = 0.0
        train_preds: list[int] = []
        train_labels: list[int] = []
        train_probs: list[np.ndarray] = []

        for waveforms, labels in tqdm(train_loader, desc=f"[{name}] train {epoch+1}/{num_epochs}", leave=False):
            waveforms = waveforms.to(device, non_blocking=True)
            raw = labels.long().to(device, non_blocking=True)  # already the CE target (0/1/2)

            optimizer.zero_grad(set_to_none=True)
            # features in fp32 OUTSIDE autocast: complex-spectrogram NaNs under fp16
            feats = forward_features(waveforms, raw, mel_train)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                outputs = model(feats)  # (B, 3) class logits; branched models unpack feats into (spec, mfcc)
                loss = criterion(outputs, raw)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            probs = torch.softmax(outputs.float(), dim=1)
            train_preds.extend(outputs.float().argmax(dim=1).cpu().numpy())
            train_labels.extend(raw.cpu().numpy())
            train_probs.extend(probs.cpu().detach().numpy())

        scheduler.step()
        tr_labels_np = np.asarray(train_labels)
        train_probs_np = np.asarray(train_probs)
        train_acc = 100.0 * float(np.mean(tr_labels_np == np.asarray(train_preds)))
        train_f1 = f1_score(tr_labels_np, train_preds, average="macro", zero_division=0)
        train_auc = _macro_auc(tr_labels_np, train_probs_np)

        model.eval()
        val_loss = 0.0
        all_preds: list[int] = []
        all_labels: list[int] = []
        all_probs: list[np.ndarray] = []

        with torch.inference_mode():
            for waveforms, labels in tqdm(val_loader, desc=f"[{name}] val {epoch+1}/{num_epochs}", leave=False):
                waveforms = waveforms.to(device, non_blocking=True)
                raw = labels.long().to(device, non_blocking=True)
                feats = forward_features(waveforms, raw, mel_eval_sep if use_sep else mel_eval)
                with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                    outputs = model(feats)
                    loss = criterion(outputs, raw)
                probs = torch.softmax(outputs.float(), dim=1)
                all_preds.extend(outputs.float().argmax(dim=1).cpu().numpy())
                all_labels.extend(raw.cpu().numpy())
                all_probs.extend(probs.cpu().detach().numpy())
                val_loss += loss.item()

        tr = np.asarray(all_labels)
        val_probs_np = np.asarray(all_probs)
        val_macro_f1 = f1_score(tr, all_preds, average="macro", zero_division=0)
        val_auc = _macro_auc(tr, val_probs_np)
        train_epoch_loss = running_loss / len(train_loader)
        val_epoch_loss = val_loss / len(val_loader)
        val_accuracy = 100.0 * float(np.mean(tr == np.asarray(all_preds)))

        history["train_loss"].append(train_epoch_loss)
        history["val_loss"].append(val_epoch_loss)
        history["val_F1"].append(val_macro_f1)
        history["val_acc"].append(val_accuracy)
        history["val_auc"].append(val_auc)
        history["train_acc"].append(train_acc)
        history["train_F1"].append(train_f1)
        history["train_auc"].append(train_auc)

        state = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "history": history,
        }
        _atomic_torch_save(state, _ckpt_path(name))
        if val_epoch_loss < best_val_loss - 1e-6:
            best_val_loss = val_epoch_loss
            plateau = 0
            _atomic_torch_save(state, _ckpt_best_path(name))
        else:
            plateau += 1

        print(f"[{name}] Epoch [{epoch+1}/{num_epochs}] -> "
              f"Train Loss: {train_epoch_loss:.4f} | Train Acc: {train_acc:.2f}% | Train F1: {train_f1:.4f} | Train AUC: {train_auc:.4f} | "
              f"Val Loss: {val_epoch_loss:.4f} | Val F1: {val_macro_f1:.4f} | Val AUC: {val_auc:.4f} | Val Acc: {val_accuracy:.2f}%")

        # epoch-15 guardrail: stop once past epoch 15 unless val loss is still improving
        if epoch >= min_epoch and plateau >= patience:
            print(f"=> [{name}] early stop @ epoch {epoch+1}: val_loss flat {patience} epochs (best {best_val_loss:.4f})")
            break

    trained_models_all[name] = model

    # ---- OOF collection: ALWAYS runs (fresh, resumed-mid, or fully-complete) ----
    best_path = _ckpt_best_path(name) if os.path.exists(_ckpt_best_path(name)) else _ckpt_path(name)
    best = torch.load(best_path, weights_only=True, map_location=device)
    model.load_state_dict(best["model_state_dict"])
    model.eval()
    oof_true: list[int] = []
    oof_probs: list[np.ndarray] = []
    with torch.inference_mode():
        for waveforms, labels in tqdm(val_loader, desc=f"[{name}] oof", leave=False):
            waveforms = waveforms.to(device, non_blocking=True)
            raw = labels.long()
            feats = forward_features(waveforms, raw, mel_eval_sep if use_sep else mel_eval)
            outputs = model(feats)
            probs = torch.softmax(outputs.float(), dim=1)
            oof_true.extend(raw.cpu().numpy())
            oof_probs.extend(probs.cpu().numpy())
    # paths come from the base indices in fold order (loader is shuffle=False)
    oof_paths = [dataset.get_path(int(i)) for i in val_subset.indices]
    oof = {
        "path": np.asarray(oof_paths, dtype=object),
        "true": np.asarray(oof_true, dtype=int),
        "probs": np.asarray(oof_probs, dtype=float),
    }
    print(f"=> [{name}] OOF collected: {len(oof_true)} segments / {len(set(oof_paths))} recordings")
    return history, oof

In [14]:
# ==== Model factory setup + separate-stream pipelines for the branched model ====
from modules.nn import BranchedCoughCNN
from modules.nn.base_nn import ResNetModel

CV_MODELS: list[str] = ["default", "branched", "resnet"]  # each model gets its own 5-fold CV run

# separate spec/mfcc streams for the branched model (pipeline returns (spec, mfcc))
mel_train_plain_sep: SpectMFCCPipeline = SpectMFCCPipeline(combind=False).to(device)
mel_eval_sep: SpectMFCCPipeline = SpectMFCCPipeline(combind=False).to(device)
mel_eval_sep = torch.compile(mel_eval_sep) if device.type == "cuda" else mel_eval_sep


In [32]:
# ---- Helpers shared by train_model: model factory, AUC, resume, OOF aggregation ----
def _make_model(model_name: str, num_classes: int = 3) -> nn.Module:
    """Model factory by string. resnet = ResNet50 ImageNet weights, head swapped, full finetune."""
    if model_name == "default":
        return DefaultCNN(num_classes)
    if model_name == "branched":
        return BranchedCoughCNN(num_classes)
    if model_name == "resnet":
        return ResNetModel(num_classes, freeze_backbone=True)
    raise ValueError(f"unknown model_name {model_name!r} (default|branched|resnet)")

def _ckpt_best_path(name: str) -> str:
    return os.path.join(RUN_STATE_DIR, f"{name}_best.pth.tar")

trained_models_all: dict[str, nn.Module] = {}  # registry _resume_with_best / train_model write into

def _safe_auc(y_true: np.ndarray, y_score: np.ndarray) -> float | None:
    """OVR AUC for one class; None when the fold is degenerate (single true class)."""
    try:
        if len(np.unique(y_true)) < 2:
            return None
        return float(roc_auc_score(y_true, y_score))
    except ValueError:
        return None

def _macro_auc(y_true: np.ndarray, probs: np.ndarray) -> float:
    """Macro one-vs-rest AUC from softmax probs (n, 3). Degenerate classes skipped;
    nan when no class is computable (kept so curves show the gap, never silently 0.5)."""
    aucs = [a for c in range(probs.shape[1])
            if (a := _safe_auc((y_true == c).astype(int), probs[:, c])) is not None]
    return float(np.mean(aucs)) if aucs else float("nan")

def multiclass_oof_metrics(rec_true: np.ndarray, rec_pred: np.ndarray, rec_probs: np.ndarray) -> dict:
    """Recording-level 3-class metrics. AUCs guarded: degenerate fold -> None."""
    cm = confusion_matrix(rec_true, rec_pred, labels=[0, 1, 2])
    recalls = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    f1s = f1_score(rec_true, rec_pred, average=None, labels=[0, 1, 2], zero_division=0)
    aucs = [_safe_auc((rec_true == c).astype(int), rec_probs[:, c]) for c in range(3)]
    macro = [a for a in aucs if a is not None]
    return {
        "macro_f1": float(f1s.mean()),
        "recall_healthy": float(recalls[0]),
        "recall_covid": float(recalls[1]),
        "recall_symptomatic": float(recalls[2]),
        "auc_healthy": aucs[0],
        "auc_covid": aucs[1],
        "auc_symptomatic": aucs[2],
        "macro_auc": float(np.mean(macro)) if macro else None,
    }

def _resume_with_best(name: str, model: nn.Module, optimizer: optim.Optimizer,
                      scheduler: optim.lr_scheduler.LRScheduler,
                      scaler: torch.amp.GradScaler, num_epochs: int) -> tuple[int, dict, float, int]:
    """Load _latest checkpoint into model/opt/sched/scaler; derive best val_loss
    and trailing plateau length from history. Returns (epochs_done, history, best_vl, plateau)."""
    best_vl, plate = float("inf"), 0
    path = _ckpt_path(name)
    fresh: dict[str, list[float]] = {"train_loss": [], "train_acc": [], "train_F1": [], "train_auc": [],
                                     "val_loss": [], "val_acc": [], "val_F1": [], "val_auc": []}
    if not os.path.exists(path):
        return 0, fresh, best_vl, plate
    ckpt = torch.load(path, weights_only=True, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    epochs_done: int = ckpt["epoch"] + 1  # 'epoch' is the 0-based index of the last COMPLETED epoch
    history: dict[str, list[float]] = ckpt["history"]
    for k in ("train_acc", "train_F1", "train_auc", "val_auc"):  # older checkpoints predate these keys
        history.setdefault(k, []).extend([float("nan")] * (len(history["val_loss"]) - len(history.get(k, []))))
    if history["val_loss"]:
        best_vl = min(history["val_loss"])
        for vl in reversed(history["val_loss"]):
            if vl <= best_vl + 1e-6:
                break
            plate += 1
    if epochs_done >= num_epochs:
        trained_models_all[name] = model
    print(f"=> [{name}] resumed at epoch {epochs_done + 1}/{num_epochs} (best val_loss {best_vl:.4f})" if epochs_done < num_epochs
          else f"=> [{name}] already complete ({num_epochs}/{num_epochs} epochs) - restored, no retrain")
    return epochs_done, history, best_vl, plate


def aggregate_recordings(oof: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    """Mean softmax probs per recording; prediction = argmax of the averaged probs."""
    order: dict[str, list[int]] = {}
    for i, p in enumerate(oof["path"]):
        order.setdefault(p, []).append(i)
    rec_true: list[int] = []
    rec_probs: list[np.ndarray] = []
    rec_paths: list[str] = []
    for path, idxs in order.items():
        labs = {int(oof["true"][i]) for i in idxs}
        assert len(labs) == 1, f"recording {path} spans labels {labs}"
        rec_paths.append(path)
        rec_true.append(labs.pop())
        rec_probs.append(np.mean(oof["probs"][idxs], axis=0))
    probs = np.asarray(rec_probs)
    return {"path": np.asarray(rec_paths, dtype=object), "true": np.asarray(rec_true, dtype=int),
            "pred": probs.argmax(axis=1), "probs": probs}

In [33]:
# ---- Augmentation-config experiment (DefaultCNN only, fixed split - NOT CV) ----
# Isolates the augmentation axis on the from-scratch baseline BEFORE any CV fold
# training. Split is recording-level via dataset_split_by_recording over ALL paths
# (NOT cell 4's segment-random dataset_split: that holdout leaks sibling segments
# across pool/test, so it must not serve as a test set). The third output here is
# this experiment's own test set.
import sys, time, contextlib

os.makedirs("tables_3class", exist_ok=True)
os.makedirs("figures_3class", exist_ok=True)
os.makedirs("logs_3class", exist_ok=True)

AUG_CONFIGS: dict[str, tuple[bool, bool]] = {
    "wave_only": (True, False),   # waveform aug only
    "spec_only": (False, True),   # spec-aug (stretch + time/freq masks) only
    "both": (True, True),         # the config the CV runs use
}

aug_train_subset, aug_val_subset, aug_test_subset = dataset_split_by_recording(
    dataset, dataset.paths, 0.8, 0.1, seed=SEED
)
print(f"aug split (recording-level): train {len(aug_train_subset.indices)} | "
      f"val {len(aug_val_subset.indices)} | test {len(aug_test_subset.indices)} segments")
_aug_tr_labels: dict[int, int] = {}
for i in aug_train_subset.indices:
    _aug_tr_labels[int(dataset.labels[i])] = _aug_tr_labels.get(int(dataset.labels[i]), 0) + 1
print("aug train segment labels:", {CLASS_NAMES[k]: v for k, v in _aug_tr_labels.items()})

@contextlib.contextmanager
def aug_log():
    """Append-mode stdout/stderr redirect to the aug experiment log; restores on exit."""
    f = open("logs_3class/aug_experiment.txt", "a", encoding="utf-8")
    out, err = sys.stdout, sys.stderr
    sys.stdout = sys.stderr = f
    try:
        yield f
    finally:
        sys.stdout, sys.stderr = out, err
        f.close()

aug split (recording-level): train 21769 | val 2725 | test 2717 segments
aug train segment labels: {'symptomatic': 4652, 'healthy': 16094, 'COVID-19': 1023}


In [34]:
# ---- Train the 3 aug configs on the fixed split (cache-proof: name-keyed checkpoints) ----
AUG_HIST_COLS: list[str] = ["train_loss", "train_acc", "train_F1", "train_auc",
                            "val_loss", "val_acc", "val_F1", "val_auc"]

def _aug_epochs_done(name: str) -> int:
    """Epochs already cached for this run (0 if no checkpoint) - the cache-proof probe."""
    p = _ckpt_path(name)
    if not os.path.exists(p):
        return 0
    return int(torch.load(p, weights_only=True, map_location="cpu")["epoch"]) + 1

_aug_orig = (train_subset, val_subset)
_aug_cached = 0
with aug_log():
    try:
        for cfg, (w, s) in AUG_CONFIGS.items():
            name = f"aug__{cfg}__default"
            done_before = _aug_epochs_done(name)
            train_subset, val_subset = aug_train_subset, aug_val_subset
            t0 = time.time()
            hist, _ = train_model(name, min_epoch=8, wave_aug=w, spec_aug=s)
            pl.DataFrame({k: hist[k] for k in AUG_HIST_COLS}).write_csv(
                f"tables_3class/{name}_experiment_stats.csv")
            print(f"=> {name}: {len(hist['val_loss'])} epochs, best val_loss "
                  f"{min(hist['val_loss']):.4f} ({time.time() - t0:.0f}s)")
            if done_before >= num_epochs:
                _aug_cached += 1
                print(f"=> cache proof: {name} served from checkpoint (no retrain)")
    finally:
        train_subset, val_subset = _aug_orig
print(f"cache: {_aug_cached}/{len(AUG_CONFIGS)} aug runs served from checkpoint")

[aug__wave_only__default] train 1/60:   0%|          | 0/378 [00:00<?, ?it/s]

[aug__wave_only__default] val 1/60:   0%|          | 0/22 [00:00<?, ?it/s]

[aug__wave_only__default] train 2/60:   0%|          | 0/378 [00:00<?, ?it/s]

[aug__wave_only__default] val 2/60:   0%|          | 0/22 [00:00<?, ?it/s]

[aug__wave_only__default] train 3/60:   0%|          | 0/378 [00:00<?, ?it/s]

[aug__wave_only__default] val 3/60:   0%|          | 0/22 [00:00<?, ?it/s]

... (66 outputs left out)

[aug__both__default] train 12/60:   0%|          | 0/378 [00:00<?, ?it/s]

[aug__both__default] val 12/60:   0%|          | 0/22 [00:00<?, ?it/s]

[aug__both__default] oof:   0%|          | 0/22 [00:00<?, ?it/s]

cache: 0/3 aug runs served from checkpoint


In [ ]:
# ---- Test-set eval per aug config (recording-level, from best checkpoints) ----
_AUG_EVAL_COLS: list[str] = ["config", "n_recordings", "macro_f1", "recall_healthy", "recall_covid",
                             "recall_symptomatic", "auc_healthy", "auc_covid", "auc_symptomatic", "macro_auc"]
aug_eval_rows: list[dict] = []
aug_test_rec: dict[str, dict[str, np.ndarray]] = {}
_test_loader = torch.utils.data.DataLoader(
    Subset(dataset, aug_test_subset.indices), BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True,
)

def _strip_compile_prefix(state: dict) -> dict:
    """torch.compile saves keys with an '_orig_mod.' prefix; the uncompiled eval
    model can't load them as-is. Strip it (no-op for non-compiled checkpoints)."""
    return {k.removeprefix("_orig_mod."): v for k, v in state.items()}

with aug_log():
    for cfg in AUG_CONFIGS:
        name = f"aug__{cfg}__default"
        aug_model = _make_model("default", num_classes=3).to(device)
        best = torch.load(_ckpt_best_path(name), weights_only=True, map_location=device)
        aug_model.load_state_dict(_strip_compile_prefix(best["model_state_dict"]))
        aug_model.eval()
        oof_true: list[int] = []
        oof_probs: list[np.ndarray] = []
        with torch.inference_mode():
            for waveforms, labels in tqdm(_test_loader, desc=f"[{name}] test", leave=False):
                waveforms = trim_tile(waveforms).to(device, non_blocking=True)
                raw = labels.long().to(device)
                feats = mel_eval(waveforms, raw)
                if feats.ndim == 3:
                    feats = feats.unsqueeze(0)
                logits = aug_model(feats)
                probs = torch.softmax(logits.float(), dim=1)
                oof_true.extend(raw.cpu().tolist())
                oof_probs.extend(probs.cpu().numpy())
        aug_oof = {
            "path": np.asarray([dataset.get_path(int(i)) for i in aug_test_subset.indices], dtype=object),
            "true": np.asarray(oof_true, dtype=int),
            "probs": np.asarray(oof_probs),
        }
        rec = aggregate_recordings(aug_oof)
        aug_test_rec[cfg] = rec
        m = multiclass_oof_metrics(rec["true"], rec["pred"], rec["probs"])
        aug_eval_rows.append({"config": cfg, "n_recordings": len(rec["true"]),
                              **{k: (round(v, 4) if v is not None else None) for k, v in m.items()}})
        cm = confusion_matrix(rec["true"], rec["pred"], labels=[0, 1, 2])
        print(f"\n=== aug test [{cfg}] ({len(rec['true'])} recordings) ===")
        print(f"(rows=true {CLASS_NAMES}, cols=pred)\n{cm}")
        print({k: m[k] for k in m})
pl.DataFrame(aug_eval_rows).write_csv("tables_3class/aug_test_eval.csv")
_tmp = "run_state_3class/aug_test_state.pt.tmp"
torch.save({"rec": aug_test_rec, "rows": aug_eval_rows}, _tmp)
os.replace(_tmp, "run_state_3class/aug_test_state.pt")
print("=> tables_3class/aug_test_eval.csv + run_state_3class/aug_test_state.pt written")

In [ ]:
# ---- Figures: aug experiment (training curves, test CMs, per-class prob hists) ----
# fresh-kernel recovery: load test state stashed by the test-eval cell
if "aug_test_rec" not in globals() or not aug_test_rec:
    _st = torch.load("run_state_3class/aug_test_state.pt", weights_only=False)
    aug_test_rec = _st["rec"]
    print("=> aug test state restored from run_state_3class/aug_test_state.pt")

_AUG_COLORS: dict[str, str] = {"wave_only": "tab:blue", "spec_only": "tab:orange", "both": "tab:green"}

# training curves: 2x2 panels, one line per config (train dashed on loss + AUC panels)
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
panels = [("val_loss", "val loss"), ("val_F1", "val macro F1"),
          ("val_auc", "val AUC (macro)"), ("val_acc", "val acc")]
for cfg in AUG_CONFIGS:
    df = pl.read_csv(f"tables_3class/aug__{cfg}__default_experiment_stats.csv")
    for ax, (col, title) in zip(axes.flat, panels):
        ax.plot(df[col].to_list(), label=cfg, color=_AUG_COLORS[cfg])
    axes.flat[0].plot(df["train_loss"].to_list(), color=_AUG_COLORS[cfg], ls="--", alpha=0.45)
    axes.flat[2].plot(df["train_auc"].to_list(), color=_AUG_COLORS[cfg], ls="--", alpha=0.45)
axes.flat[0].set_title("loss (dashed = train)")
axes.flat[2].set_title("AUC macro OVR (dashed = train)")
for i, ax in enumerate(axes.flat):
    ax.set_ylabel(panels[i][1]); ax.set_xlabel("epoch")
fig.legend(*axes.flat[0].get_legend_handles_labels(), loc="upper right")
fig.tight_layout()
fig.savefig("figures_3class/aug_default_curves.png", dpi=200)
plt.show(fig); plt.close(fig)

# test confusion matrices (recording level, counts)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, cfg in zip(axes, AUG_CONFIGS):
    rec = aug_test_rec[cfg]
    cm = confusion_matrix(rec["true"], rec["pred"], labels=[0, 1, 2])
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{cfg} - test recordings")
fig.tight_layout()
fig.savefig("figures_3class/aug_default_test_cm.png", dpi=200)
plt.show(fig); plt.close("all")

# P(true class) per recording by true class (the honest signal picture)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, cfg in zip(axes, AUG_CONFIGS):
    rec = aug_test_rec[cfg]
    for c in range(3):
        ax.hist(rec["probs"][rec["true"] == c, c], bins=30, alpha=0.5, range=(0, 1), label=CLASS_NAMES[c])
    ax.set_title(cfg); ax.set_xlabel("P(true class)")
axes[0].set_ylabel("recordings"); axes[0].legend()
fig.tight_layout()
fig.savefig("figures_3class/aug_default_probs_hist.png", dpi=200)
plt.show(fig); plt.close(fig)
print("aug figures written to figures_3class/")

In [ ]:
# ---- 5-fold CV: 3-class models, pooled OOF recording-level metrics ----
# Every pool recording is validated exactly once across the folds. Each fold's
# best-val-loss checkpoint feeds pooled out-of-fold segment softmax probs, which
# are averaged per RECORDING before any metric is computed (segment-level
# numbers are never the headline). Logs append to logs_3class/cv_experiment.txt.
import sys, time, atexit
from datetime import datetime
from zoneinfo import ZoneInfo

LOG_PATH_CV = r"logs_3class/cv_experiment.txt"
os.makedirs("logs_3class", exist_ok=True)
_logf_cv = open(LOG_PATH_CV, "a", encoding="utf-8")
sys.stdout = _logf_cv
sys.stderr = _logf_cv
atexit.register(_logf_cv.close)

print(f"\n\n\n TIME OF RUN: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}")

HIST_COLS: list[str] = ["train_loss", "train_acc", "train_F1", "train_auc",
                        "val_loss", "val_acc", "val_F1", "val_auc"]

oof_all: dict[str, dict[str, np.ndarray]] = {}  # model -> pooled per-segment OOF across folds
cv_rows: list[dict] = []
_orig = (train_subset, val_subset)
try:
    for mn in CV_MODELS:
        seg_acc: dict[str, list] = {"path": [], "true": [], "probs": [], "fold": []}
        for fi, (tr_pos, va_pos) in enumerate(FOLDS):
            train_subset = Subset(dataset, [cv_pool[p] for p in tr_pos])
            val_subset = Subset(dataset, [cv_pool[p] for p in va_pos])
            key = f"cv__{mn}__f{fi}"
            print(f"\n=== fold {fi + 1}/{len(FOLDS)}  {mn} ===")
            t0 = time.time()
            hist, oof = train_model(key, model_name=mn)
            best_ep = int(np.argmin(hist["val_loss"]))
            cv_rows.append({
                "model": mn, "fold": fi, "epochs": len(hist["val_loss"]),
                "val_loss": round(hist["val_loss"][best_ep], 4),
                "val_acc": round(hist["val_acc"][best_ep], 4),
                "val_F1": round(hist["val_F1"][best_ep], 4),
                "train_auc": round(hist["train_auc"][best_ep], 4),
                "val_auc": round(hist["val_auc"][best_ep], 4),
            })
            print(f"=> {key}: best val_loss {hist['val_loss'][best_ep]:.4f} @ epoch {best_ep + 1} ({time.time() - t0:.0f}s)")
            seg_acc["path"].extend(oof["path"])
            seg_acc["true"].extend(oof["true"])
            seg_acc["probs"].extend(oof["probs"])
            seg_acc["fold"].extend([fi] * len(oof["probs"]))
            pl.DataFrame({k: hist[k] for k in HIST_COLS}).write_csv(f"tables_3class/cv__{mn}__f{fi}_experiment_stats.csv")
        oof_all[mn] = {k: np.asarray(v) for k, v in seg_acc.items()}
finally:
    train_subset, val_subset = _orig

# persist OOF state so eval/figures survive a fresh kernel
def _save_oof_state(obj: dict) -> None:
    tmp = "run_state_3class/oof_state.pt.tmp"
    torch.save(obj, tmp)
    os.replace(tmp, "run_state_3class/oof_state.pt")

_save_oof_state({"oof_all": oof_all, "cv_rows": cv_rows})
print("=> OOF state saved to run_state_3class/oof_state.pt")
pl.DataFrame(cv_rows).write_csv("tables_3class/cv_folds.csv")
print("=> wrote cv_folds.csv")

In [ ]:
# ---- Recording-level OOF eval per model (the headline numbers) ----
# fresh-kernel recovery: load OOF state persisted by the CV cell
if "oof_all" not in globals() or not oof_all:
    _st = torch.load("run_state_3class/oof_state.pt", weights_only=False)
    oof_all, cv_rows = _st["oof_all"], _st["cv_rows"]
    print("=> OOF state restored from run_state_3class/oof_state.pt")

# aggregate_recordings lives in the helpers cell (shared with the aug test cell)
EVAL_COLS: list[str] = ["model", "n_recordings", "macro_f1", "recall_healthy", "recall_covid", "recall_symptomatic",
                        "auc_healthy", "auc_covid", "auc_symptomatic", "macro_auc"]
eval_rows: list[dict] = []
rec_level: dict[str, dict[str, np.ndarray]] = {}
for mn in CV_MODELS:
    rec = aggregate_recordings(oof_all[mn])
    rec_level[mn] = rec
    m = multiclass_oof_metrics(rec["true"], rec["pred"], rec["probs"])
    eval_rows.append({"model": mn, "n_recordings": len(rec["true"]),
                      **{k: (round(v, 4) if v is not None else None) for k, v in m.items()}})
    cm = confusion_matrix(rec["true"], rec["pred"], labels=[0, 1, 2])
    print(f"\n=== OOF recording-level [{mn}] ({len(rec['true'])} recordings) ===")
    print(f"(rows=true {CLASS_NAMES}, cols=pred)\n{cm}")
    print({k: m[k] for k in m})
pl.DataFrame(eval_rows).write_csv("tables_3class/oof_eval.csv")
print("=> wrote oof_eval.csv")

In [ ]:
# ---- Figures: per-fold histories, 3-class CMs, per-class prob distributions ----
os.makedirs("figures_3class", exist_ok=True)

for mn in CV_MODELS:
    # history: one val-loss line per fold
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plotted = False
    for fi in range(N_FOLDS):
        csv = f"tables_3class/cv__{mn}__f{fi}_experiment_stats.csv"
        if os.path.exists(csv):
            df = pl.read_csv(csv)
            ax.plot(df["val_loss"].to_list(), label=f"fold {fi}")
            plotted = True
    if plotted:
        ax.set_title(f"{mn}: val loss per CV fold"); ax.set_xlabel("epoch"); ax.legend()
        fig.tight_layout(); fig.savefig(f"figures_3class/{mn}_history.png", dpi=200)
        plt.show(fig); plt.close(fig)

    # confusion matrix (recording level)
    rec = rec_level[mn]
    cm = confusion_matrix(rec["true"], rec["pred"], labels=[0, 1, 2])
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(cmap="Blues")
    plt.title(f"{mn} - OOF recording-level confusion matrix")
    plt.savefig(f"figures_3class/{mn}_cm.png", bbox_inches="tight", dpi=200)
    plt.show(); plt.close("all")

    # honest signal picture: per-recording P(true class) by true class
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for c in range(3):
        ax.hist(rec["probs"][rec["true"] == c, c], bins=30, alpha=0.5, label=CLASS_NAMES[c], range=(0, 1))
    ax.set_title(f"{mn}: per-recording P(true class) by true class"); ax.set_xlabel("P(true class)"); ax.legend()
    fig.tight_layout(); fig.savefig(f"figures_3class/{mn}_probs_hist.png", dpi=200)
    plt.show(fig); plt.close(fig)
print("figures written to figures_3class/")